In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install --upgrade pymupdf
!pip install numpy==2.0.2

import subprocess
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True)
    HAS_GPU = result.returncode == 0
except FileNotFoundError:
    HAS_GPU = False

if HAS_GPU:
    !python -m pip install paddlepaddle-gpu==3.2.2 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    !pip install fastapi uvicorn opencv-python insightface onnxruntime-gpu faiss-cpu python-multipart sentence-transformers pandas pyarrow
else:
    !python -m pip install paddlepaddle==3.2.2
    !pip install fastapi uvicorn opencv-python insightface onnxruntime faiss-cpu python-multipart sentence-transformers pandas pyarrow

!pip install "paddleocr>=3.0.0"
!apt-get update && apt-get install -y libgl1-mesa-glx

In [ ]:
BASE_DIR = "/content/drive/My Drive/face-recog-runner"
BASE_DIR_WITHOUT_SPACE = "/content/drive/MyDrive/face-recog-runner"

In [ ]:
# ============================================================
# PARTNER CONFIG — chỉ cần edit dòng này
# ============================================================
CURRENT_USER = "tuta1w1aa0000@gmail.com"

ALL_USERS = [
    "tuta1w1aa0000@gmail.com",
    # "vanhuy2710@gmail.com",
    # "nhathuyvo83@gmail.com",
    # "dtdat.1117@gmail.com",
    # "thanhdatsilver@gmail.com",
    # "thuyvipham1995@gmail.com",
    # "vi.pham@vidratek.com",
    # "viviheo1995@gmail.com",
    # "huy.vanminh@returnself.studio",
    # "lamhai.lhh997@gmail.com",
]

In [ ]:
!pip install langchain

In [ ]:
!nvcc --version

In [ ]:
!pip install "paddleocr>=3.0.0"

In [ ]:
import os
import cv2
import numpy as np
import json
import faiss
# OOM fix -  sử dụng gabarage collection + pytorch để giải phóng RAM/VRAM
import gc
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
import pandas as pd
import uuid
from typing import List, Dict, Optional, Tuple
from glob import glob
from tqdm import tqdm
import insightface
import paddle
import paddleocr
from paddleocr import PaddleOCR
from sentence_transformers import SentenceTransformer


In [ ]:
try:
    print(f"PaddlePaddle Version: {paddle.__version__}")
    print(f"PaddleOCR Version: {paddleocr.VERSION if hasattr(paddleocr, 'VERSION') else 'Unknown'}")
    print(f"NumPy Version: {np.__version__}")

    # Verify Paddle GPU
    paddle.utils.run_check()
    print("PaddlePaddle GPU check passed.")
except Exception as e:
    print(f"WARNING: GPU check failed or version mismatch: {e}")

In [ ]:
class Settings:
    # InsightFace
    DET_SIZE = (640, 640)
    MODEL_ROOT = os.path.join(BASE_DIR, "data/insightface_models")

    # FAISS
    VECTOR_DIM = 512
    INDEX_PATH = os.path.join(BASE_DIR, "data/index/index.faiss")
    METADATA_PATH = os.path.join(BASE_DIR, "data/index/metadata.parquet")

    # OCR & Text Search
    BIB_INDEX_PATH = os.path.join(BASE_DIR, "data/index/bib_index.faiss")
    BIB_METADATA_PATH = os.path.join(BASE_DIR, "data/index/bib_metadata.parquet")
    TEXT_EMBEDDING_MODEL = "all-MiniLM-L6-v2"
    TEXT_EMBEDDING_DIM = 384

    # Video Processing
    FRAME_INTERVAL = 30
    VIDEO_DIR = os.path.join(BASE_DIR, "data/videos")

settings = Settings()


In [ ]:
# Ensure directories exist
os.makedirs(os.path.dirname(settings.INDEX_PATH), exist_ok=True)
os.makedirs(settings.MODEL_ROOT, exist_ok=True)
os.makedirs(settings.VIDEO_DIR, exist_ok=True)

In [ ]:
class FaceTracker:
    def __init__(self, similarity_threshold: float = 0.6, max_missed_frames: int = 2):
        self.active_tracks = []
        self.similarity_threshold = similarity_threshold
        self.max_missed_frames = max_missed_frames

    def _cosine_similarity(self, emb1: np.ndarray, emb2: np.ndarray) -> float:
        norm1 = np.linalg.norm(emb1)
        norm2 = np.linalg.norm(emb2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        return np.dot(emb1, emb2) / (norm1 * norm2)

    def update(self, faces: List[Dict], frame_idx: int, timestamp: float) -> List[Dict]:
        finished_tracks = []

        if not faces:
            for track in self.active_tracks:
                track['missed_frames'] += 1
        else:
            matches = []
            used_tracks = set()
            used_faces = set()

            for t_idx, track in enumerate(self.active_tracks):
                track_emb = track['best_face']['embedding']
                for f_idx, face in enumerate(faces):
                    score = self._cosine_similarity(track_emb, face['embedding'])
                    if score > self.similarity_threshold:
                        matches.append((t_idx, f_idx, score))

            matches.sort(key=lambda x: x[2], reverse=True)

            for t_idx, f_idx, score in matches:
                if t_idx in used_tracks or f_idx in used_faces:
                    continue

                track = self.active_tracks[t_idx]
                face = faces[f_idx]

                track['missed_frames'] = 0
                track['last_seen_frame_idx'] = frame_idx
                track['end_timestamp'] = timestamp
                # OOM fix: cái này không dùng
                # track['history'].append(face)

                if face['det_score'] > track['best_face']['det_score']:
                     track['best_face'] = face

                used_tracks.add(t_idx)
                used_faces.add(f_idx)

            for f_idx, face in enumerate(faces):
                if f_idx not in used_faces:
                     self.active_tracks.append({
                         'id': str(uuid.uuid4()),
                         'best_face': face,
                         'start_timestamp': timestamp,
                         'end_timestamp': timestamp,
                         'last_seen_frame_idx': frame_idx,
                         'missed_frames': 0
                        # OOM fix: cái này không dùng
                        #  'history': [face]
                     })

            for t_idx, track in enumerate(self.active_tracks):
                if t_idx not in used_tracks:
                    track['missed_frames'] += 1

        new_active = []
        for track in self.active_tracks:
            if track['missed_frames'] > self.max_missed_frames:
                finished_tracks.append(track)
            else:
                new_active.append(track)

        self.active_tracks = new_active
        return finished_tracks

    def finalize(self) -> List[Dict]:
        finished = list(self.active_tracks)
        self.active_tracks = []
        return finished

In [ ]:
class FaceProcessor:
    def __init__(self):
        self.app = insightface.app.FaceAnalysis(name='buffalo_l', root=settings.MODEL_ROOT)
        self.app.prepare(ctx_id=0, det_size=settings.DET_SIZE)

    def get_embedding(self, img_array: np.ndarray) -> List[dict]:
        faces = self.app.get(img_array)
        results = []
        for face in faces:
             results.append({
                 'embedding': face.embedding,
                 'bbox': face.bbox.astype(int).tolist(),
                 'det_score': float(face.det_score)
             })
        return results


In [ ]:
class OCRProcessor:
    def __init__(self, lang: str = 'en'):
        self.ocr = PaddleOCR(lang=lang)

    def get_text_v1(self, img_array: np.ndarray) -> List[Dict]:
        """
        Detect text in an image and return list of results.
        Result format: [{'text': '...', 'bbox': [[x1,y1], [x2,y2], ...], 'score': ...}]
        """
        # PaddleOCR expects image in RGB or BGR? OpenCV reads BGR. PaddleOCR handles it.
        # result = self.ocr.ocr(img_array, cls=True)
        # result is a list of lists (one for each image if batch, but here we pass one image)

        result = self.ocr.ocr(img_array, cls=True)

        output = []
        if result and result[0]:
            for line in result[0]:
                # line format: [points, (text, score)]
                # points: [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
                points = line[0]
                text, score = line[1]

                output.append({
                    'text': text,
                    'bbox': points, # List of 4 points
                    'score': score
                })
        return output

    def get_text(self, img_array: np.ndarray) -> List[Dict]:
        result = self.ocr.ocr(img_array)
        output = []
        if not result:
            return output

        res = result[0]
        rec_texts = res.get('rec_texts', [])
        rec_scores = res.get('rec_scores', [])
        rec_polys = res.get('rec_polys', [])

        for text, score, bbox in zip(rec_texts, rec_scores, rec_polys):
            output.append({
                'text': text,
                'bbox': bbox.tolist() if hasattr(bbox, 'tolist') else bbox,
                'score': score
            })
        return output

In [ ]:
class VectorStore:
    def __init__(self, load_existing: bool = True):
        self.dimension = settings.VECTOR_DIM
        self.indices = []
        self.bib_indices = []

        if load_existing:
            self.load()
        else:
            self.indices.append((faiss.IndexFlatIP(self.dimension), []))
            self.bib_indices.append((faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM), []))

    @property
    def index(self):
        return self.indices[0][0]

    @property
    def metadata(self):
        return self.indices[0][1]

    @property
    def bib_index(self):
        return self.bib_indices[0][0]

    @property
    def bib_metadata(self):
        return self.bib_indices[0][1]

    def add_vectors(self, vectors: np.ndarray, metadata_entries: List[Dict]):
        if vectors.shape[0] != len(metadata_entries):
            raise ValueError("Number of vectors and metadata entries must match.")
        faiss.normalize_L2(vectors)
        self.index.add(vectors)
        self.metadata.extend(metadata_entries)

    def add_bib_vectors(self, vectors: np.ndarray, metadata_entries: List[Dict]):
        if vectors.shape[0] != len(metadata_entries):
            raise ValueError("Number of vectors and metadata entries must match.")
        faiss.normalize_L2(vectors)
        self.bib_index.add(vectors)
        self.bib_metadata.extend(metadata_entries)

    def save(self):
        os.makedirs(os.path.dirname(settings.INDEX_PATH), exist_ok=True)
        faiss.write_index(self.index, settings.INDEX_PATH)

        if self.metadata:
            df = pd.DataFrame(self.metadata)
            df.to_parquet(settings.METADATA_PATH, index=False)

        if self.bib_indices:
            faiss.write_index(self.bib_index, settings.BIB_INDEX_PATH)
            if self.bib_metadata:
                df = pd.DataFrame(self.bib_metadata)
                df.to_parquet(settings.BIB_METADATA_PATH, index=False)

    def load(self):
        print("Loading indices...")
        self.indices = []
        self.bib_indices = []

        shard_pattern = os.path.join(os.path.dirname(settings.INDEX_PATH), "shard_*")
        shard_dirs = sorted(glob(shard_pattern), key=lambda x: x)

        if shard_dirs:
            print(f"Found {len(shard_dirs)} shards: {shard_dirs}")
            for d in shard_dirs:
                self._load_pair(d)
        else:
            if os.path.exists(settings.INDEX_PATH):
                print(f"Loading main index from {settings.INDEX_PATH}")
                self._load_pair(os.path.dirname(settings.INDEX_PATH), main_file=True)
            else:
                self.indices.append((faiss.IndexFlatIP(self.dimension), []))
                self.bib_indices.append((faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM), []))

    def _load_pair(self, directory: str, main_file: bool = False):
        if main_file:
            idx_path = settings.INDEX_PATH
            meta_path = settings.METADATA_PATH
            bib_idx_path = settings.BIB_INDEX_PATH
            bib_meta_path = settings.BIB_METADATA_PATH
        else:
            idx_path = os.path.join(directory, "index.faiss")
            meta_path = os.path.join(directory, "metadata.parquet")
            bib_idx_path = os.path.join(directory, "bib_index.faiss")
            bib_meta_path = os.path.join(directory, "bib_metadata.parquet")

        try:
            if os.path.exists(idx_path):
                idx = faiss.read_index(idx_path)
                meta = []
                if os.path.exists(meta_path):
                    try:
                        df = pd.read_parquet(meta_path)
                        meta = json.loads(df.to_json(orient='records'))
                    except Exception as e:
                        print(f"[LOAD_PAIR] error reading metadata parquet: {e}")
                self.indices.append((idx, meta))
        except Exception as e:
            print(f"Error loading face index ({idx_path}): {e}")

        try:
            if os.path.exists(bib_idx_path):
                bib_idx = faiss.read_index(bib_idx_path)
                bib_meta = []
                if os.path.exists(bib_meta_path):
                    try:
                        df = pd.read_parquet(bib_meta_path)
                        bib_meta = json.loads(df.to_json(orient='records'))
                    except Exception as e:
                        print(f"Error loading bib index ({bib_idx_path}): {e}")
                self.bib_indices.append((bib_idx, bib_meta))
            else:
                self.bib_indices.append((faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM), []))
        except Exception as e:
            print(f"Error loading bib index ({bib_idx_path}): {e}")
            self.bib_indices.append((faiss.IndexFlatIP(settings.TEXT_EMBEDDING_DIM), []))

In [ ]:
# Utils Funtions
def update_sample_interval(
    current_faces: list,
    no_face_streak: int,
    process_interval: int
) -> tuple[int, int]:
    """
    Adaptive sampling: giảm interval khi có face, tăng khi không có face liên tiếp.
    Returns: (new_process_interval, new_no_face_streak)
    """
    if current_faces:
        return max(settings.FRAME_INTERVAL // 2, 5), 0
    else:
        new_streak = no_face_streak + 1
        if new_streak >= 3:
            return min(settings.FRAME_INTERVAL * 2, 90), new_streak
        return process_interval, new_streak

In [ ]:
def process_video(video_path: str, face_processor: FaceProcessor, ocr_processor: OCRProcessor, text_model: SentenceTransformer, vector_store: VectorStore, face_tracker: FaceTracker):
    print(f"Processing {video_path}...")
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return

    frame_count = 0
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    process_interval = settings.FRAME_INTERVAL
    no_face_streak = 0
    next_process_frame = 0  # [Sampling frame]

    pbar = tqdm(total=total_frames)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        pbar.update(1)

        # [Sampling frame] Skip frame nếu chưa đến next_process_frame
        if frame_count < next_process_frame:
            del frame
            continue

        # [Sampling frame] Cập nhật next checkpoint
        next_process_frame = frame_count + process_interval

        timestamp = frame_count / fps

        # --- Face Detection & Tracking ---
        current_faces = []
        try:
            faces = face_processor.get_embedding(frame)
            if faces:
                current_faces = faces
        except Exception as e:
            print(f"Error processing face frame {frame_count}: {e}")

        # [Sampling frame] Adaptive interval
        process_interval, no_face_streak = update_sample_interval(
            current_faces, no_face_streak, process_interval
        )

        finished_tracks = face_tracker.update(current_faces, frame_count, timestamp)

        if finished_tracks:
            embeddings = []
            metadata_list = []
            for track in finished_tracks:
                best_face = track['best_face']
                embeddings.append(best_face['embedding'])
                metadata_list.append({
                    'video_path': video_path.replace(BASE_DIR, "data"),
                    'timestamp': track['start_timestamp'],
                    'end_timestamp': track['end_timestamp'],
                    'frame_idx': track['last_seen_frame_idx'],
                    'bbox': best_face['bbox'],
                    'det_score': best_face['det_score'],
                    'type': 'face',
                    'track_id': track['id']
                })

            if embeddings:
                vector_store.add_vectors(np.array(embeddings), metadata_list)

        # --- OCR / Bib Detection ---
        if current_faces:
            try:
                for face in current_faces:
                    top_faces = sorted(current_faces, key=lambda f: f['det_score'], reverse=True)[:5]
                    fx1, fy1, fx2, fy2 = [int(v) for v in face['bbox']]
                    fw = fx2 - fx1
                    fh = fy2 - fy1

                    frame_h, frame_w = frame.shape[:2]

                    cx1 = max(0, int(fx1 - fw * 0.5))
                    cx2 = min(frame_w, int(fx2 + fw * 0.5))
                    cy1 = min(frame_h, fy2)
                    cy2 = min(frame_h, int(fy2 + fh * 3.0))

                    if cx2 <= cx1 or cy2 <= cy1:
                        continue

                    chest_crop = frame[cy1:cy2, cx1:cx2]

                    target_width = 320
                    scale = 1.0
                    if (cx2 - cx1) > target_width:
                        scale = target_width / (cx2 - cx1)
                        new_h = int((cy2 - cy1) * scale)
                        chest_crop = cv2.resize(chest_crop, (target_width, new_h))

                    texts = ocr_processor.get_text(chest_crop)

                    if texts:
                        bib_embeddings = []
                        bib_metadata_list = []

                        for item in texts:
                            text_str = item['text']
                            if any(c.isdigit() for c in text_str):

                                emb = text_model.encode([text_str])[0]

                                orig_bbox = []
                                for pt in item['bbox']:
                                    px, py = pt
                                    orig_x = int(cx1 + px / scale)
                                    orig_y = int(cy1 + py / scale)
                                    orig_bbox.append([orig_x, orig_y])

                                bib_embeddings.append(emb)
                                bib_metadata_list.append({
                                    'video_path': video_path.replace(BASE_DIR, "data"),
                                    'timestamp': timestamp,
                                    'frame_idx': frame_count,
                                    'bbox': orig_bbox,
                                    'score': item['score'],
                                    'text': text_str,
                                    'type': 'bib'
                                })

                        if bib_embeddings:
                            vector_store.add_bib_vectors(np.array(bib_embeddings), bib_metadata_list)

            except Exception as e:
                print(f"Error processing OCR for face in frame {frame_count}: {e}")

    # FORCE FINISH remaining tracks
    remaining_tracks = face_tracker.finalize()
    if remaining_tracks:
        embeddings = []
        metadata_list = []
        for track in remaining_tracks:
            best_face = track['best_face']
            embeddings.append(best_face['embedding'])
            metadata_list.append({
                'video_path': video_path.replace(BASE_DIR, "data"),
                'timestamp': track['start_timestamp'],
                'end_timestamp': track['end_timestamp'],
                'frame_idx': track['last_seen_frame_idx'],
                'bbox': best_face['bbox'],
                'det_score': best_face['det_score'],
                'type': 'face',
                'track_id': track['id']
            })
        if embeddings:
            vector_store.add_vectors(np.array(embeddings), metadata_list)

    cap.release()
    pbar.close()


def load_processed_videos(processed_path: str) -> set:
    if os.path.exists(processed_path):
        with open(processed_path, 'r') as f:
            return set(json.load(f))
    return set()


def save_processed_video(processed_path: str, processed_videos: set):
    with open(processed_path, 'w') as f:
        json.dump(list(processed_videos), f)

In [ ]:
# ============================================================
# REDISTRIBUTE V2 — chia pending videos đều cho tất cả users
# Note: Cần load cell settings ở trên trước rồi mới load cell này
# ============================================================
RUN_REDISTRIBUTE_V2 = False

BATCH_SIZE = 5

def redistribute_v2():
    from glob import glob
    index_dir = os.path.join(BASE_DIR, "data")
    video_dir = os.path.join(BASE_DIR, "data/videos")

    all_videos = sorted(
        glob(os.path.join(video_dir, "*.mp4")) +
        glob(os.path.join(video_dir, "*.MP4")) +
        glob(os.path.join(video_dir, "*.avi")) +
        glob(os.path.join(video_dir, "*.mov"))
    )

    # 1. Đọc từ assignment files hiện tại
    already_done = set()
    for f in glob(os.path.join(index_dir, "assignment-*.json")):
        with open(f, 'r') as fp:
            records = json.load(fp)
        for r in records["videos"]:
            if r["done"]:
                already_done.add(r["path"])

    # 2. (Removable) Merge với processed_videos.json cũ nếu có
    checkpoint_path = os.path.join(BASE_DIR, "data/processed_videos.json")
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r') as f:
            already_done.update(json.load(f))

    print(f"Total: {len(all_videos)} | Done: {len(already_done)} | Pending: {len(all_videos) - len(already_done)}")

    # 3. Xoá file cũ
    for f in glob(os.path.join(index_dir, "assignment-*.json")):
        os.remove(f); print(f"Removed: {f}")

    n = len(ALL_USERS)
    slice_size = len(all_videos) // n
    remainder = len(all_videos) % n

    assignment = {}
    start = 0
    for i, email in enumerate(ALL_USERS):
        extra = 1 if i < remainder else 0
        end = start + slice_size + extra
        chunk = all_videos[start:end]

        shard_offset = start // BATCH_SIZE

        records = {
            "shard_offset": shard_offset,
            "videos": [{"path": v, "done": v in already_done} for v in chunk]
        }

        safe_email = email.replace("@", "_").replace(".", "_")
        fname = f"assignment-{safe_email}.json"
        fpath = os.path.join(index_dir, fname)
        with open(fpath, 'w') as f:
            json.dump(records, f, indent=2)

        assignment[email] = fname
        done_count = sum(1 for v in chunk if v in already_done)
        print(f"{email}: {len(chunk)} videos, shard_offset={shard_offset}, {done_count} done ({fname})")
        start = end

    with open(os.path.join(index_dir, "assignment.json"), 'w') as f:
        json.dump(assignment, f, indent=2)
    print("\nDone.")

if RUN_REDISTRIBUTE_V2:
    redistribute_v2()

In [ ]:
# ============================================================
# AUTO-LOAD ASSIGNMENT
# ============================================================
def update_my_assignment(video_path):
    index_dir = os.path.join(BASE_DIR, "data")
    assignment_path = os.path.join(index_dir, "assignment.json")

    with open(assignment_path, 'r') as f:
        assignment = json.load(f)

    fname = assignment[CURRENT_USER]
    fpath = os.path.join(index_dir, fname)

    with open(fpath, 'r') as f:
        records = json.load(f)

    for r in records["videos"]:
        if r["path"] == video_path:
            r["done"] = True
            break

    with open(fpath, 'w') as f:
        json.dump(records, f, indent=2)

def load_my_assignment():
    index_dir = os.path.join(BASE_DIR, "data")
    assignment_path = os.path.join(index_dir, "assignment.json")

    if not os.path.exists(assignment_path):
        raise FileNotFoundError("assignment.json không tồn tại. Owner cần chạy cell Redistribute trước.")

    with open(assignment_path, 'r') as f:
        assignment = json.load(f)

    if CURRENT_USER not in assignment:
        raise ValueError(f"{CURRENT_USER} không có trong assignment.")

    fname = assignment[CURRENT_USER]
    fpath = os.path.join(index_dir, fname)

    with open(fpath, 'r') as f:
        records = json.load(f)

    shard_offset = records["shard_offset"]
    my_videos = [r["path"] for r in records["videos"]]
    already_done = set(r["path"] for r in records["videos"] if r["done"])
    pending = [r["path"] for r in records["videos"] if not r["done"]]

    print(f"User        : {CURRENT_USER}")
    print(f"Assignment  : {fname} ({len(my_videos)} videos)")
    print(f"Shard offset: {shard_offset}")
    print(f"Done        : {len(already_done)}")
    print(f"Pending     : {len(pending)} videos")
    return my_videos, already_done, shard_offset

MY_VIDEOS, GLOBAL_DONE, MY_SHARD_OFFSET = load_my_assignment()

MAIN

In [ ]:
def main():
    BATCH_SIZE = 5

    print(f"BASE_DIR : {BASE_DIR}")
    print(f"USER     : {CURRENT_USER}")

    face_processor = FaceProcessor()
    ocr_processor = OCRProcessor()
    text_model = SentenceTransformer(settings.TEXT_EMBEDDING_MODEL)

    video_files = MY_VIDEOS
    processed_videos = set(GLOBAL_DONE)

    if not video_files:
        print("Không có video nào được assign.")
        return

    pending = [v for v in video_files if v not in processed_videos]
    print(f"Assigned: {len(video_files)} | Pending: {len(pending)}")

    for i in range(0, len(video_files), BATCH_SIZE):
        batch_idx = MY_SHARD_OFFSET + (i // BATCH_SIZE)
        batch_videos = video_files[i : i + BATCH_SIZE]

        if all(v in processed_videos for v in batch_videos):
            print(f"Batch {batch_idx} already done, skipping.")
            continue

        shard_dir = os.path.join(BASE_DIR, f"data/index/shard_{batch_idx}")
        os.makedirs(shard_dir, exist_ok=True)

        settings.INDEX_PATH = os.path.join(shard_dir, "index.faiss")
        settings.METADATA_PATH = os.path.join(shard_dir, "metadata.parquet")
        settings.BIB_INDEX_PATH = os.path.join(shard_dir, "bib_index.faiss")
        settings.BIB_METADATA_PATH = os.path.join(shard_dir, "bib_metadata.parquet")

        shard_has_updates = False
        print(f"\n--- Batch {batch_idx}: {len(batch_videos)} videos ---")

        for video_file in batch_videos:
            if video_file in processed_videos:
                print(f"Skipping: {video_file}")
                continue

            vector_store = VectorStore(load_existing=True)
            face_tracker = FaceTracker()
            process_video(video_file, face_processor, ocr_processor, text_model, vector_store, face_tracker)

            vector_store.save()

            processed_videos.add(video_file)
            update_my_assignment(video_file)

            del vector_store
            gc.collect()
            if HAS_TORCH:
                torch.cuda.empty_cache()

            shard_has_updates = True
            print(f"Saved: {video_file}")

        if shard_has_updates:
            print(f"Batch {batch_idx} saved to {shard_dir}")
        else:
            print(f"Batch {batch_idx} already done.")

    print("\nAll assigned videos processed.")

main()
